In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

# Thread-safe counter
count = 0
count_lock = Lock()

def Cancel_Remove(driver):
    try:
        button = driver.find_element(By.CSS_SELECTOR, 'div[data-auto-updatecity-cancel="true"]')
        button.click()
    except:
        pass

def Data_extraction(soup):
    ProductName, MarketerName, Uses, Features, Directions, Safety = [], [], [], [], [], []

    desc_div = soup.find("div", class_="ProductDescription__description-content___A_qCZ")
    if desc_div:
        title_tag = desc_div.find("strong")
        if title_tag:
            ProductName.append(title_tag.get_text(strip=True))

        headings = desc_div.find_all("strong")
        for heading in headings:
            heading_text = heading.get_text(strip=True)
            items = []

            for elem in heading.next_siblings:
                if getattr(elem, 'name', None) == "strong":
                    break
                if elem.name == "ul":
                    for li in elem.find_all("li"):
                        items.append(li.get_text(strip=True))
                elif elem.name == "p":
                    text = elem.get_text(" ", strip=True)
                    if text:
                        items.append(text)
                elif getattr(elem, 'name', None) is None:
                    text = str(elem).strip()
                    if text:
                        items.append(text)

            if "Uses" in heading_text or "Concerns It Helps With" in heading_text:
                Uses.extend(items)
            elif ("Product Specifications" in heading_text or "Key Ingredients" in heading_text or
                  "Product Form" in heading_text or "Net Quantity" in heading_text or
                  "Suitable For" in heading_text or "Key Benefits" in heading_text):
                Features.extend(items)
            elif "Directions" in heading_text:
                Directions.extend(items)
            elif "Safety" in heading_text:
                Safety.extend(items)

    marketer_div = soup.find("div", class_="ProductTitle__marketer___7Wsj9")
    if marketer_div:
        marketer_tag = marketer_div.find("a")
        if marketer_tag:
            MarketerName.append(marketer_tag.get_text(strip=True))

    return ProductName, MarketerName, Uses, Features, Directions, Safety

def scrape_url(url):
    global count
    CHROME_DRIVER_PATH = os.path.join(os.getcwd(), 'chromedriver.exe')
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_experimental_option("excludeSwitches", ["enable-logging"])

    service = Service(CHROME_DRIVER_PATH)
    driver = webdriver.Chrome(service=service, options=options)

    try:
        driver.get(url)
        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.ProductDescription__description-content___A_qCZ"))
        )
        Cancel_Remove(driver)
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        ProductName, MarketerName, Uses, Features, Directions, Safety = Data_extraction(soup)

        # Mandatory check only for MarketerName
        if not MarketerName:
            raise ValueError("MarketerName not found")

        with count_lock:
            count += 1
            row_number = count

        row = {
            "row_number": row_number,
            "url": url,
            "ProductName": ProductName[0] if ProductName else "",
            "MarketerName": MarketerName[0],
            "Uses": ", ".join(Uses) if Uses else "",
            "Features": ", ".join(Features) if Features else "",
            "Directions": ", ".join(Directions) if Directions else "",
            "Safety": ", ".join(Safety) if Safety else ""
        }

        print(f"Row {row_number} Done: {row['MarketerName']} - {row['ProductName']}")
        return row

    except Exception as e:
        print(f"Error for {url}: {e}")
        return None
    finally:
        driver.quit()


def scrape_urls_multithread(urls, max_threads=3, save_every=100, save_path=None):
    all_data = []
    df_final = pd.DataFrame()
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        results = executor.map(scrape_url, urls)
        for i, r in enumerate(results, 1):
            if r:
                all_data.append(r)

            # Save every 'save_every' URLs
            if i % save_every == 0 and save_path:
                df_batch = pd.DataFrame(all_data)
                df_final = pd.concat([df_final, df_batch], ignore_index=True)
                df_final.to_csv(save_path, index=False)
                all_data = []  # reset batch
                print(f"Saved {i} records to CSV.")

    # Save any remaining data
    if all_data and save_path:
        df_batch = pd.DataFrame(all_data)
        df_final = pd.concat([df_final, df_batch], ignore_index=True)
        df_final.to_csv(save_path, index=False)
        print(f"Saved final batch, total {len(df_final)} records.")

    return df_final

# Example usage
df_urls = pd.read_csv(r"c:\\Users\\MICILMEDS\Documents\\Medi_final\\correct_data\\Category\\supports-braces.csv")
urls = df_urls.iloc[4000:, 5].tolist()  # adjust column index for URL
df = scrape_urls_multithread(
    urls, 
    max_threads=5, 
    save_every=100,
    save_path=r"c:\\Users\\MICILMEDS\Documents\\Medi_final\\correct_data\\Category\\Scraped_Supports_Braces_Final_4.csv"
)


Row 1 Done: IGR - IGR Knee Immobilizer 26 Inch Grey XL
Row 2 Done: Alnacare Ortho - 
Row 3 Done: Ascent Meditech Limited - Flamingo Premium Below Knee Stockings Medium Latex Free
Row 4 Done: Health Point - Health Point WH-917 Ankle Brace with Double String Strap Wrapping
Row 5 Done: Medtrix - 
Row 6 Done: PR Flexmake Private Limited - 
Row 7 Done: BeatXP - beatXP Knee Cap
Row 8 Done: BeatXP - beatXP Lumbo Sacral Support Belt for Tummy Reduction
Row 9 Done: IGR - IGR Easy Ankle Comfort Blue Large
Row 10 Done: Tynor Orthotics Pvt. Ltd. - Tynor Elbow Support Air Pro
Row 11 Done: RCSP - RCSP Cervical Collar Skin Colour
Row 12 Done: MPK I Gate Pvt. Ltd. - Ion Clad Copper Compression Calf Sleeve
Row 13 Done: Witzion - Witzion Lumbo Sacral Back Support Belt
Row 14 Done: PR Flexmake Private Limited - 
Row 15 Done: Global Axis Alliance - BESAFE Forever Knee Cap Support Band
Row 16 Done: Vissco Rehabilitation Aids Pvt. Ltd. - Vissco Core 1421 Knee Support with Velcro (Neoprene)
Row 17 Done: Unit